# NotebookPrecision

Precision/Recall workflow for generated images vs real dataset features.

This notebook was updated to use `UnifiedDatasetLoader` (no hardcoded CIFAR batch-file paths).

In [ ]:
# MODIFIED: dynamic repo-root detection so this notebook runs from repo root or Notebooks/.
from __future__ import annotations

import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, SubsetRandomSampler
from torchvision import models, transforms

CURRENT = Path.cwd().resolve()
if (CURRENT / 'Datasets').exists():
    REPO_ROOT = CURRENT
elif (CURRENT.parent / 'Datasets').exists():
    REPO_ROOT = CURRENT.parent
else:
    raise RuntimeError('Could not locate repository root from current working directory.')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Datasets.dataset_subset import DatasetSubsetConfig
from Datasets.unified_dataset_loader import DatasetConfig, UnifiedDatasetLoader
from Models.dcgan import DCGANGenerator
from Metrics.precision_recall import compute_precision_recall

print(f'Repo root: {REPO_ROOT}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


In [ ]:
# MODIFIED: dataset is now configured through UnifiedDatasetLoader, not hardcoded file paths.
DATASET_NAME = 'cifar10'  # change to: mnist | cifar10 | celeba | chestxray14
DATA_ROOTS = {
    'mnist': str(REPO_ROOT / 'data' / 'MNIST'),
    'cifar10': str(REPO_ROOT / 'data' / 'CIFAR10'),
    'celeba': str(REPO_ROOT / 'data' / 'CelebA'),
    'chestxray14': str(REPO_ROOT / 'data' / 'ChestXray14'),
}
IMAGE_SIZE = 32 if DATASET_NAME in {'mnist', 'cifar10'} else 64
BATCH_SIZE = 32
REAL_SUBSET_SIZE = 500
SEED = 42

# Optional dataset subsetting before DataLoader sampling.
subset_cfg = DatasetSubsetConfig(
    max_samples=None,
    fraction=None,
    seed=SEED,
    strategy='random',
)

loader_cfg = DatasetConfig(
    name=DATASET_NAME,
    data_root=DATA_ROOTS[DATASET_NAME],
    image_size=IMAGE_SIZE,
    normalize_to_neg_one_one=True,
    subset_config=subset_cfg,
)
unified_loader = UnifiedDatasetLoader(loader_cfg)

try:
    train_ds = unified_loader.get_dataset(train=True, download=False)
except (FileNotFoundError, RuntimeError) as exc:
    print(f'Local dataset missing/incomplete ({exc}); trying download/setup...')
    train_ds = unified_loader.get_dataset(train=True, download=True)

print(f'Train dataset size: {len(train_ds)}')
class_names = list(getattr(train_ds, 'classes', []))
if class_names:
    print(f'Classes: {class_names}')


In [ ]:
# MODIFIED: visualize samples from UnifiedDatasetLoader dataset object.
rng = np.random.default_rng(SEED)
sample_count = min(10, len(train_ds))
sample_indices = rng.choice(len(train_ds), size=sample_count, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.flat, sample_indices):
    image, label = train_ds[int(idx)]
    img = image.detach().cpu()
    if img.shape[0] == 1:
        img = img.repeat(3, 1, 1)
    img = ((img + 1.0) / 2.0).clamp(0.0, 1.0).permute(1, 2, 0).numpy()
    ax.imshow(img)
    if isinstance(label, torch.Tensor) and label.ndim == 0:
        label_idx = int(label.item())
        label_text = class_names[label_idx] if class_names and label_idx < len(class_names) else str(label_idx)
    else:
        label_text = 'multi-label'
    ax.set_title(label_text)
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# MODIFIED: configurable random subset sampler from loaded dataset for feature extraction.
real_subset_size = min(REAL_SUBSET_SIZE, len(train_ds))
real_indices = np.random.default_rng(SEED).choice(len(train_ds), size=real_subset_size, replace=False)
real_sampler = SubsetRandomSampler(real_indices.tolist())
real_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=real_sampler, num_workers=0)
print(f'Real subset size: {real_subset_size}')


In [ ]:
# MODIFIED: checkpoint path is configurable (no hardcoded netG path in code).
GENERATOR_CHECKPOINT = REPO_ROOT / 'outputs' / 'dcgan_cifar10' / 'netG_latest.pth'
LATENT_DIM = 100
NGF = 64
FAKE_BATCH_SIZE = 32
FAKE_NUM_BATCHES = 16

sample_x, _ = train_ds[0]
num_channels = int(sample_x.shape[0])
generator = DCGANGenerator(ngpu=0, nc=num_channels, nz=LATENT_DIM, ngf=NGF, image_size=IMAGE_SIZE).to(device)

if not GENERATOR_CHECKPOINT.exists():
    raise FileNotFoundError(f'Generator checkpoint not found: {GENERATOR_CHECKPOINT}')
generator.load_state_dict(torch.load(GENERATOR_CHECKPOINT, map_location=device))
generator.eval()
print(f'Loaded generator from: {GENERATOR_CHECKPOINT}')


In [ ]:
# MODIFIED: shared preprocessing for InceptionV3 features (works for real + generated tensors).
inception = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)
inception.fc = nn.Identity()
inception.aux_logits = False
inception.eval().to(device)

imagenet_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
resize_299 = transforms.Resize((299, 299))

def prepare_for_inception(images: torch.Tensor) -> torch.Tensor:
    x = images.to(device)
    if x.ndim != 4:
        raise ValueError('Expected BCHW tensor.')
    if x.shape[1] == 1:
        x = x.repeat(1, 3, 1, 1)
    # Convert from [-1, 1] (loader normalization) to [0, 1].
    x = ((x + 1.0) / 2.0).clamp(0.0, 1.0)
    x = resize_299(x)
    x = (x - imagenet_mean) / imagenet_std
    return x

def extract_features_from_loader(loader: DataLoader) -> np.ndarray:
    features = []
    with torch.no_grad():
        for batch in loader:
            images = batch[0] if isinstance(batch, (tuple, list)) else batch
            processed = prepare_for_inception(images)
            feats = inception(processed)
            features.append(feats.detach().cpu().numpy())
    return np.concatenate(features, axis=0)

def extract_features_from_generator(generator_model: torch.nn.Module, num_batches: int, batch_size: int) -> np.ndarray:
    features = []
    with torch.no_grad():
        for _ in range(num_batches):
            z = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
            fake = generator_model(z)
            processed = prepare_for_inception(fake)
            feats = inception(processed)
            features.append(feats.detach().cpu().numpy())
    return np.concatenate(features, axis=0)


In [ ]:
real_features = extract_features_from_loader(real_loader)
fake_features = extract_features_from_generator(generator, num_batches=FAKE_NUM_BATCHES, batch_size=FAKE_BATCH_SIZE)

print('real_features:', real_features.shape)
print('fake_features:', fake_features.shape)


In [ ]:
results = compute_precision_recall(real_features, fake_features, k=3)
print(results)
